### Store Your API Key Securely
* Store API key as a Databricks secret:

In [0]:
# dbutils.widgets.text("openweather_api_key", "")
# dbutils.widgets.text("rapidapi_api_key", "")
# dbutils.widgets.text("serpapi_api_key", "")

### Weather Function
* get_store_weather function added:

In [0]:
openweather_api_key = dbutils.widgets.get("openweather_api_key")
rapidapi_api_key = dbutils.widgets.get("openweather_api_key")
serpapi_api_key = dbutils.widgets.get("openweather_api_key")

In [0]:
%sql
    
-- Weather Function with OpenWeather API
-- Returns "No weather data available" if API fails

CREATE OR REPLACE FUNCTION accenture.sales_analysis.get_store_weather(
    store_location STRING
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Get real-time weather from OpenWeather API - No fallback data'
AS $$
import requests
import json

def get_weather(location):
    """
    Get current weather from OpenWeather API
    Returns actual weather or "No weather data available" message
    No fallback/cached data
    """

    # OpenWeather Current Weather API endpoint
    # Using Current Weather Data API (not One Call API)
    # Docs: https://openweathermap.org/current
    url = f"http://api.openweathermap.org/data/2.5/weather"
    
    # Parameters
    params = {
        'q': f"{location},US",  # City name + country code
        'appid': '$openweather_api_key',
        'units': 'imperial'  # Fahrenheit
    }
    
    try:
        # Make API request with 10 second timeout
        response = requests.get(url, params=params, timeout=10)
        
        # Check if request was successful (200 OK)
        if response.status_code == 200:
            data = response.json()
            
            # Extract weather information
            temp = data['main']['temp']
            feels_like = data['main']['feels_like']
            condition = data['weather'][0]['main']
            description = data['weather'][0]['description']
            humidity = data['main']['humidity']
            wind_speed = data['wind']['speed']
            
            # Format response with actual data
            weather_info = (
                f"Weather in {location}: {condition} ({description}), "
                f"Temp: {temp:.0f}°F (feels like {feels_like:.0f}°F), "
                f"Humidity: {humidity}%, Wind: {wind_speed:.1f} mph"
            )
            
            return weather_info
            
        elif response.status_code == 404:
            # City not found
            return f"No weather data available - Location '{location}' not found"
            
        elif response.status_code == 401:
            # Invalid API key
            return "No weather data available - API authentication failed"
            
        elif response.status_code == 429:
            # Rate limit exceeded
            return "No weather data available - API rate limit exceeded"
            
        else:
            # Other API errors
            return f"No weather data available - API error (status: {response.status_code})"
    
    except requests.exceptions.Timeout:
        return "No weather data available - API request timeout"
    
    except requests.exceptions.ConnectionError:
        return "No weather data available - Unable to connect to weather service"
    
    except requests.exceptions.RequestException as e:
        return f"No weather data available - Network error"
    
    except KeyError as e:
        return "No weather data available - Unexpected API response format"
    
    except Exception as e:
        return f"No weather data available - Error: {type(e).__name__}"

return get_weather(store_location)
$$;

In [0]:
%sql
SELECT accenture.sales_analysis.get_store_weather('Detroit, MI');

In [0]:
%sql
CREATE OR REPLACE FUNCTION accenture.sales_analysis.get_competitor_price(
    product_name STRING
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Get competitor pricing from RapidAPI Real-Time Product Search'
AS $$
import requests
import json

def get_price(product):
    """
    Get current market prices from RapidAPI
    Returns competitor pricing or error message
    """
    
    # RapidAPI Product Search endpoint
    url = "https://real-time-product-search.p.rapidapi.com/search-v2"
    
    # Query parameters
    querystring = {
        "q": product,
        "country": "us",
        "language": "en",
        "page": "1",
        "limit": "5",
        "sort_by": "BEST_MATCH",
        "product_condition": "ANY"
    }
    
    # Headers - REPLACE WITH YOUR RAPIDAPI KEY
    headers = {
        "x-rapidapi-key": '$rapidapi_api_key',
        "x-rapidapi-host": "real-time-product-search.p.rapidapi.com"
    }
    
    try:
        # Make API request
        response = requests.get(url, headers=headers, params=querystring, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            products = data.get('data', {}).get('products', [])
            
            if not products:
                return f"No pricing data found for '{product}'"
            
            # Extract pricing information
            results = []
            prices = []
            
            for item in products[:5]:
                offer = item.get('offer', {})
                price_str = offer.get('price', 'N/A')
                
                title = item.get('product_title', 'N/A')
                store = offer.get('store_name', 'N/A')
                
                results.append(f"{title} - {price_str} at {store}")
                
                # Extract numeric price
                if price_str and price_str != 'N/A':
                    try:
                        clean_price = price_str.replace('$', '').replace(',', '')
                        prices.append(float(clean_price))
                    except:
                        pass
            
            # Calculate average
            avg_price = sum(prices) / len(prices) if prices else 0
            
            # Format output
            output = f"Pricing for '{product}': Avg ${avg_price:.2f}\n"
            for i, result in enumerate(results[:3], 1):
                output += f"{i}. {result}\n"
            
            return output
            
        elif response.status_code == 403:
            return "No pricing data available - API authentication failed"
        elif response.status_code == 429:
            return "No pricing data available - API rate limit exceeded"
        else:
            return f"No pricing data available - API error (status: {response.status_code})"
    
    except requests.exceptions.Timeout:
        return "No pricing data available - API request timeout"
    except requests.exceptions.ConnectionError:
        return "No pricing data available - Unable to connect to pricing service"
    except Exception as e:
        return f"No pricing data available - Error: {type(e).__name__}"

return get_price(product_name)
$$;

In [0]:
%sql
SELECT accenture.sales_analysis.get_competitor_price('chocolate bar');

In [0]:
%sql
CREATE OR REPLACE FUNCTION accenture.sales_analysis.get_market_trends(
    query STRING
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Search Google for market research and trends via SerpAPI'
AS $$
import requests
import json

def search_google(search_query):
    """
    Perform Google Search via SerpAPI
    Returns search results or error message
    """
    
    # SerpAPI Google Search endpoint
    url = "https://serpapi.com/search"
    
    # Parameters - REPLACE WITH YOUR SERPAPI KEY
    params = {
        "engine": "google",
        "q": search_query,
        "location": "United States",
        "google_domain": "google.com",
        "gl": "us",
        "hl": "en",
        "api_key": "$serpapi_api_key"
    }
    
    try:
        # Make API request
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            organic_results = data.get("organic_results", [])
            
            if not organic_results:
                return f"No search results found for '{search_query}'"
            
            # Format results
            output = f"Search results for '{search_query}':\n\n"
            
            for i, item in enumerate(organic_results[:5], 1):
                title = item.get("title", "N/A")
                snippet = item.get("snippet", "N/A")
                link = item.get("link", "N/A")
                
                output += f"- {title}: {snippet}\n"
            
            return output
            
        elif response.status_code == 401:
            return "No search results available - Invalid API key"
        elif response.status_code == 429:
            return "No search results available - API rate limit exceeded"
        else:
            return f"No search results available - API error (status: {response.status_code})"
    
    except requests.exceptions.Timeout:
        return "No search results available - API request timeout"
    except requests.exceptions.ConnectionError:
        return "No search results available - Unable to connect to search service"
    except Exception as e:
        return f"No search results available - Error: {type(e).__name__}"

return search_google(query)
$$;

In [0]:
%sql
SELECT accenture.sales_analysis.get_market_trends('candy market trends 2025');

In [0]:
%sql
CREATE OR REPLACE FUNCTION accenture.sales_analysis.get_retail_news(
    topic STRING
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Search Google News for industry updates via SerpAPI'
AS $$
import requests
import json

def search_news(news_topic):
    """
    Search Google News via SerpAPI
    Returns news results or error message
    """
    
    # SerpAPI Google News endpoint
    url = "https://serpapi.com/search"
    
    # Parameters - REPLACE WITH YOUR SERPAPI KEY
    params = {
        "engine": "google_news",
        "q": news_topic,
        "gl": "us",
        "hl": "en",
        "api_key": "$serpapi_api_key"
    }
    
    try:
        # Make API request
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            news_results = data.get("news_results", [])
            
            if not news_results:
                return f"No news found for '{news_topic}'"
            
            # Format results
            output = f"News for '{news_topic}':\n\n"
            
            for i, item in enumerate(news_results[:5], 1):
                title = item.get("title", "N/A")
                source = item.get("source", {})
                source_name = source.get("name", "N/A") if isinstance(source, dict) else "N/A"
                date = item.get("date", "N/A")
                
                output += f"* {title} (Source: {source_name})\n"
            
            return output
            
        elif response.status_code == 401:
            return "No news available - Invalid API key"
        elif response.status_code == 429:
            return "No news available - API rate limit exceeded"
        else:
            return f"No news available - API error (status: {response.status_code})"
    
    except requests.exceptions.Timeout:
        return "No news available - API request timeout"
    except requests.exceptions.ConnectionError:
        return "No news available - Unable to connect to news service"
    except Exception as e:
        return f"No news available - Error: {type(e).__name__}"

return search_news(topic)
$$;

In [0]:
%sql
SELECT accenture.sales_analysis.get_retail_news('chocolate industry news');

In [0]:
%sql
CREATE OR REPLACE FUNCTION accenture.sales_analysis.find_competitor_locations(
    query STRING,
    location STRING
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Search Google Maps for business locations via SerpAPI'
AS $$
import requests
import json

def search_maps(search_query, search_location):
    """
    Search Google Maps via SerpAPI
    Returns location results or error message
    """
    
    # SerpAPI Google Maps endpoint
    url = "https://serpapi.com/search"
    
    # Parameters - REPLACE WITH YOUR SERPAPI KEY
    params = {
        "engine": "google_maps",
        "q": f"{search_query} {search_location}",
        "type": "search",
        "api_key": "$serpapi_api_key"
    }
    
    try:
        # Make API request
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            local_results = data.get("local_results", [])
            
            if not local_results:
                return f"No locations found for '{search_query}' in '{search_location}'"
            
            # Format results
            output = f"Locations for '{search_query}' in {search_location}:\n\n"
            
            for i, item in enumerate(local_results[:5], 1):
                title = item.get("title", "N/A")
                rating = item.get("rating", "N/A")
                reviews = item.get("reviews", "N/A")
                address = item.get("address", "N/A")
                
                output += f"{title} at {address} (Rating: {rating})\n"
            
            return output
            
        elif response.status_code == 401:
            return "No location data available - Invalid API key"
        elif response.status_code == 429:
            return "No location data available - API rate limit exceeded"
        else:
            return f"No location data available - API error (status: {response.status_code})"
    
    except requests.exceptions.Timeout:
        return "No location data available - API request timeout"
    except requests.exceptions.ConnectionError:
        return "No location data available - Unable to connect to maps service"
    except Exception as e:
        return f"No location data available - Error: {type(e).__name__}"

return search_maps(query, location)
$$;

In [0]:
%sql
SELECT accenture.sales_analysis.find_competitor_locations('candy stores', 'Austin, TX');